In [24]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

Read the tickers from a file

In [26]:
tickers = pd.read_csv('Tickers_Example.csv', header=None)[0].drop_duplicates()
type_s= []
currency = []
market = []
sector = []

for ticker in tickers: 
    try: 
        tick = yf.Ticker(ticker)
        type_s.append(tick.info.get('quoteType'))
        currency.append(tick.info.get('currency'))
        market.append(tick.info.get('market'))
        sector.append(tick.info.get('sector'))
    except:
        type_s.append(None)
        currency.append(None)
        market.append(None)
        sector.append(None)

info = {
    'Type' : type_s,
    'Currency' : currency,
    'Market' : market,
    'Sector' : sector
}

all_stocks = pd.DataFrame(info, index=tickers)
all_stocks.index.name = 'Ticker'
all_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AGN,NONE,None,None,None
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials


check if it is eligible

In [28]:
mask = all_stocks['Type'] == 'EQUITY'
mask1 = all_stocks['Currency'].isin(['USD', 'CAD'])
mask2 = all_stocks['Market'].isin(['us_market', 'ca_market'])

eligible_stocks = all_stocks[mask & mask1 & mask2]
eligible_stocks

,Type,Currency,Market,Sector
Ticker,,,,
AAPL,EQUITY,USD,us_market,Technology
ABBV,EQUITY,USD,us_market,Healthcare
ABT,EQUITY,USD,us_market,Healthcare
ACN,EQUITY,USD,us_market,Technology
AIG,EQUITY,USD,us_market,Financial Services
AMZN,EQUITY,USD,us_market,Consumer Cyclical
AXP,EQUITY,USD,us_market,Financial Services
BA,EQUITY,USD,us_market,Industrials
BAC,EQUITY,USD,us_market,Financial Services


check for volume

In [29]:
start='2025-10-01'
end='2026-09-30'
min_avg_volume=5000

eligible_stocks = eligible_stocks.copy()

for ticker in eligible_stocks.index.tolist():
    tick = yf.Ticker(ticker)
    history = tick.history(start=start, end=end)
    history['Month'] = history.index.strftime('%Y-%m')
    
    month_counts = history.groupby('Month').size()
    valid_months = month_counts[month_counts >= 18].index
    history = history[history['Month'].isin(valid_months)]

    eligible_stocks.loc[ticker, 'Avg_Volume'] = history['Volume'].mean()

mask = eligible_stocks['Avg_Volume']>=min_avg_volume
eligible_stocks = eligible_stocks[mask]

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume
Ticker,,,,,
AAPL,EQUITY,USD,us_market,Technology,48275207.6
ABBV,EQUITY,USD,us_market,Healthcare,6289325.6
ABT,EQUITY,USD,us_market,Healthcare,10094284.8
ACN,EQUITY,USD,us_market,Technology,6098460.8
AIG,EQUITY,USD,us_market,Financial Services,4303847.2
AMZN,EQUITY,USD,us_market,Consumer Cyclical,46544059.2
AXP,EQUITY,USD,us_market,Financial Services,3049604.4
BA,EQUITY,USD,us_market,Industrials,7123268.8
BAC,EQUITY,USD,us_market,Financial Services,36744519.2


take the closing price

In [34]:
#CHANGE DATE LATER TO NOVEMBER 27, 2026
buy_date = '2026-10-27'
end_date = '2026-10-28'
for ticker in eligible_stocks.index.tolist():
    tick = yf.Ticker(ticker)
    history = tick.history(start=buy_date, end=end_date)
    if not history.empty:
        eligible_stocks.loc[ticker, 'Close'] = history.loc[buy_date, 'Close']
eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD
Ticker,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,48275207.6,346.836487,486.008102
ABBV,EQUITY,USD,us_market,Healthcare,6289325.6,286.390015,401.306877
ABT,EQUITY,USD,us_market,Healthcare,10094284.8,103.099998,144.469906
ACN,EQUITY,USD,us_market,Technology,6098460.8,235.160004,329.520311
AIG,EQUITY,USD,us_market,Financial Services,4303847.2,79.580002,111.512275
AMZN,EQUITY,USD,us_market,Consumer Cyclical,46544059.2,253.100006,354.658919
AXP,EQUITY,USD,us_market,Financial Services,3049604.4,312.910004,438.468277
BA,EQUITY,USD,us_market,Industrials,7123268.8,168.179993,235.663900
BAC,EQUITY,USD,us_market,Financial Services,36744519.2,50.680000,71.015858


convert to CAD

In [35]:
usdcad = yf.Ticker('USDCAD=X').history(start=buy_date, end=end_date)
rate = usdcad.loc[buy_date, 'Close']
for ticker in eligible_stocks.index.tolist():
    price = eligible_stocks.loc[ticker, 'Close']
    if eligible_stocks.loc[ticker, 'Currency'] == 'USD':
        eligible_stocks.loc[ticker, 'Close_CAD'] = price * rate
    else :
        eligible_stocks.loc[ticker, 'Close_CAD'] = price

eligible_stocks

,Type,Currency,Market,Sector,Avg_Volume,Close,Close_CAD
Ticker,,,,,,,
AAPL,EQUITY,USD,us_market,Technology,48275207.6,346.836487,486.008102
ABBV,EQUITY,USD,us_market,Healthcare,6289325.6,286.390015,401.306877
ABT,EQUITY,USD,us_market,Healthcare,10094284.8,103.099998,144.469906
ACN,EQUITY,USD,us_market,Technology,6098460.8,235.160004,329.520311
AIG,EQUITY,USD,us_market,Financial Services,4303847.2,79.580002,111.512275
AMZN,EQUITY,USD,us_market,Consumer Cyclical,46544059.2,253.100006,354.658919
AXP,EQUITY,USD,us_market,Financial Services,3049604.4,312.910004,438.468277
BA,EQUITY,USD,us_market,Industrials,7123268.8,168.179993,235.663900
BAC,EQUITY,USD,us_market,Financial Services,36744519.2,50.680000,71.015858


fee application function

In [33]:
fee_rate = 0.001 
def calc_fee_cad(shares_num):
    return min(shares_num * fee_rate, 2.15) * rate

calc_fee_cad(10)

np.float64(0.01401260018348694)